# 리포트 06 — 실측 계획과 검증

> ### ❓ 이 편이 답하는 질문
> **무엇을 재면 시뮬레이션의 어느 주장이 결판나는가?**

### 결론
1. 야외 비행시험은 **탐지통계**를 준다. **절대 σ 는 주지 않는다** — 같은 세션의 교정표적·배경차감·자세통제·원거리장이 추가로 있어야 나온다(§2).
2. 원거리장 요구는 최대 24.4 m ⟨outputs/report06_derived.json : farfield_adopted.R_ff_max_m⟩ (Matrice 4E · WiFi, 로터 디스크 포함 외접 대각 기준).
3. 주파수 기울기를 판정하려면 세션간 진폭 재현성이 2.44 dB ⟨outputs/report06_derived.json : slope.gap_db_min⟩ 보다 좋아야 한다.
4. 앵커의 최대 미해소 항인 크기법칙(L² vs L⁴, 9.50 dB ⟨outputs/report06_derived.json : size_law.uncontrolled_size_db⟩)은 두 기체를 함께 재면 4.06 dB ⟨outputs/report06_derived.json : size_law.differential_db⟩ 의 차등신호로 갈린다(§4).
5. 12-bit ADC 여유는 최소 19.2 dB ⟨outputs/report06_measurement.json : adc.headroom_db_min⟩ — 자유공간 기하 기준이고 야외 직접파에서는 줄어든다(§1).

### ✅ 주장하는 것 / ❌ 주장하지 않는 것

| ✅ 이 편이 주장하는 것 | ❌ 이 편이 주장하지 않는 것 |
|---|---|
| 절대 σ 를 얻는 데 **무엇이 필요한가** — 6가지 요구조건과 각각의 수치 임계 | **측정 결과** — 이 편에는 잰 값이 하나도 없다. 전부 설계값이다 |
| 원거리장·각도표본·점표적·교정구·지면반사 분리의 **계산된 설계값** | 야외 비행시험만으로 절대 σ 가 나온다는 것 — 교정 없이는 상대값뿐이다 |
| 시뮬의 주장마다 **어느 측정이 결판내는가**(§4 결정표) | 시뮬 σ 의 정확도 — 이 편은 그것을 검증하지 않는다(02편이 경계를 긋는다) |
| 장비 제약이 무엇을 막는가 — X410 12-bit ADC 여유 | 편파에 대한 예측 — 커널에 편파 자유도가 없어 예측 자체가 불가능하다 |
|  | 야외 절대 탐지거리의 시뮬 대조 — 환경이 다르다(§3) |

### 필요한 사전지식

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 02 §4 | 앵커가 무엇을 통제하고 무엇을 통제하지 못하는가 |
| 03 | 세 조명원(LTE·5G·WiFi)의 대역과 점유 |
| 04 | 명목 Pfa 와 경험 Pfa 를 왜 교정해야 하는가 |
| 05 | 자유공간에서 나온 탐지 결과가 무엇을 가정하는가 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (GPU 불필요) |
| 비고 | 긴 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 에 있고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---

## §1. 하드웨어 — X410 한 대로 기준과 감시를 동시에

TX 4채널 · RX 4채널이다. **RX0 = 기준 채널**(직접파), **RX1~3 = 감시 배열**로 쓴다.
사양은 `src/experiment_x410.py:60`, 배치는 `src/experiment_x410.py:100` 한 곳에만 있다.

⚠ **12-bit ADC 가 야외의 천장이다.** 직접파를 양자화하는 순간 그 아래는 못 지운다.
그 한계 모형이 `src/experiment_x410.py:83` 의 `adc_quantize()` 다.

| 항목 | 값 | 무엇을 제약하나 |
|---|---|---|
| TX / RX 채널 | 4 ⟨outputs/report06_measurement.json : hw.n_tx⟩ / 4 ⟨outputs/report06_measurement.json : hw.n_rx⟩ | 기준 1 + 감시 3 배분 |
| 채널당 순시대역 | 400 MHz ⟨outputs/report06_measurement.json : hw.max_bw_mhz⟩ | 거리분해능과 점표적 조건(§2-3) |
| 주파수 범위 | 1 MHz ⟨outputs/report06_derived.json : hw_span.f_lo_mhz⟩ ~ 7.2 GHz ⟨outputs/report06_derived.json : hw_span.f_hi_ghz⟩ | 세 밴드 전부 커버 |
| ADC 동적범위 | 74.01 dB ⟨outputs/report06_measurement.json : hw.dynamic_range_db⟩ | 직접파 제거의 천장 |
| 감시배열 AoA 빔폭 | 33.8° ⟨outputs/report06_measurement.json : hw.aoa_beamwidth_deg⟩ | 각도 관측 — 디텍션에는 불필요 |
| 최대대역 바이스태틱 ΔR | 0.749 m ⟨outputs/report06_measurement.json : hw.range_res_bistatic_m_at_max_bw⟩ | 표적이 여러 거리빈에 퍼진다(§2-3) |

원사양 출처는 `src/experiment_x410.py:61 (ni.com / ettus.com 2024 spec)` 한 곳이다.

![adc headroom](outputs/figures/report06_adc_headroom.png)

**그림 1.** 12-bit ADC 는 직접파 대 잡음비 위에 얼마의 여유를 남기는가?

가장 빡빡한 파형은 `LTE20` 이다 — 점유대역이 좁아 기준채널 이득이 높다.

⚠ 이 DNR 은 **자유공간 시뮬 기하**의 값이다⟨outputs/report06_measurement.json : adc.dnr_source⟩. 야외에서 송수신을 가깝게 놓으면 DNR 이 올라가 여유가 줄어든다.

## §2. ⭐ 탐지성능과 절대 σ 는 다른 실험이다

야외 비행시험은 **탐지통계**(검출·오경보·최대거리)를 준다. σ 는 주지 않는다.
σ 는 절대량이라 같은 세션 안에 기준이 있어야 하고, 자세와 환경이 통제되어야 한다.

| 야외 비행시험이 이미 주는 것 | 절대 σ 가 **추가로** 요구하는 것 |
|---|---|
| 표적 유/무의 검출·오경보 통계 | 같은 세션·같은 지지대의 **알려진 σ 교정표적** |
| 같은 기하에서의 파형 간 상대 우열 | 표적을 치운 **배경의 코히런트 차감** |
| 거리·도플러 응답의 존재 | 엔코더 턴테이블로 **통제된 자세** |
| 링크가 닫히는지 여부 | **원거리장** 거리 또는 근접장 서브밴드 처리 |
| 실제 클러터에서의 오경보율 | 안테나 패턴·송수신 체인 이득 교정 |

다섯 줄 전부 조건까지 적힌 산문판이 `docs/MEASUREMENT_PLAN.md` §1-1~1-6 에 있다.

### §2-1. 원거리장 — 얼마나 멀리 떨어져야 하나

`R_ff = 2D²/λ`. **D 정의를 섞으면 요구거리가 최대 3.65 ⟨outputs/report06_derived.json : farfield_adopted.spread_ratio_max⟩배 틀린다.**
채택은 가장 보수적인 정의다 — 회전 로터 디스크까지 포함한 외접상자의 3D 대각.

| 기체 | 밴드 | λ | D_env | **R_ff(env)** | R_ff(bbox) | R_ff(모터대각) |
|---|---|---|---|---|---|---|
| matrice4e | LTE | 163 mm | 0.839 m | 8.65 m | 4.42 m | 2.37 m |
| matrice4e | 5G | 86 mm | 0.839 m | 16.42 m | 8.38 m | 4.50 m |
| matrice4e | WiFi | 58 mm | 0.839 m | 24.44 m | 12.48 m | 6.69 m |
| mini5pro | LTE | 163 mm | 0.497 m | 3.03 m | 1.74 m | 0.93 m |
| mini5pro | 5G | 86 mm | 0.497 m | 5.76 m | 3.30 m | 1.77 m |
| mini5pro | WiFi | 58 mm | 0.497 m | 8.57 m | 4.91 m | 2.63 m |

출처 ⟨outputs/report06_derived.json : farfield⟩

![farfield](outputs/figures/report06_farfield.png)

**그림 2.** 각 기체와 밴드에서 원거리장에 들어가려면 얼마나 멀어야 하는가?

최대 요구는 24.44 m ⟨outputs/report06_derived.json : farfield_adopted.R_ff_max_m⟩ 다.
교정구는 기체보다 요구거리가 낮으므로, 기체 기준 거리에 같은 자리로 놓으면 자동 만족이다.

### §2-2. 교정 기준체 — σ 를 절대량으로 만드는 유일한 장치

**정밀 PEC 구**를 쓴다. 구는 방위무관이라 정렬 오차가 σ 에 안 들어간다.

기준값은 πr²(광학 점근)이 아니라 **정확 Mie** 다.
단일 출처는 `benchmark/mie_pec_sphere.py:207` 이고 자체검증 `selfcheck()` 을 갖고 있다.

세션 **시작과 끝에 한 번씩** 잰다. 두 값의 차이가 그 세션의 드리프트 예산이다.
그 차이가 목표 정확도보다 크면 그 세션 자료는 버린다.

| 구 | 밴드 | ka | σ_Mie | Mie−πr² | Matrice 4E 대비 여유 | Mini 5 Pro 대비 여유 |
|---|---|---|---|---|---|---|
| r=17.8 cm | LTE | 6.9 | -9.95 dBsm | +0.07 dB | +4.38 dB | +8.44 dB |
| r=17.8 cm | 5G | 13.1 | -9.71 dBsm | +0.31 dB | +4.27 dB | +8.33 dB |
| r=17.8 cm | WiFi | 19.4 | -9.89 dBsm | +0.13 dB | +3.73 dB | +7.79 dB |
| r=25.0 cm | LTE | 9.7 | -6.47 dBsm | +0.60 dB | +7.86 dB | +11.92 dB |
| r=25.0 cm | 5G | 18.3 | -7.06 dBsm | +0.01 dB | +6.93 dB | +10.98 dB |
| r=25.0 cm | WiFi | 27.3 | -7.14 dBsm | -0.07 dB | +6.49 dB | +10.55 dB |

출처 ⟨outputs/report06_derived.json : calibration⟩

![calibration](outputs/figures/report06_calibration.png)

**그림 3.** 어느 반경의 교정구가 세 밴드 모두에서 기체 예상 σ 위에 있는가?

교정구는 기체보다 최소 3.73 dB ⟨outputs/report06_derived.json : calibration_margin_min_db⟩ 밝다. 여유가 양수여야 같은 이득 설정으로 둘 다 잡히고, 그래야 비율이 σ 비율이 된다.

채택 반경은 17.8 cm ⟨outputs/report06_derived.json : calibration_pick.radius_cm⟩ 다 — 세 밴드 모두 Mie−πr² 편차가 작고 앵커 문헌(Yuan)이 쓴 것과 같은 크기다.

### §2-3. 점표적 조건 — 순시대역을 통째로 쓰면 안 된다

peak |s|² 를 σ 로 쓰려면 표적이 **한 거리빈 안**에 들어와야 한다.
두 기체를 함께 만족시키는 최대 서브밴드는 200 MHz ⟨outputs/report06_derived.json : point_target_max_bw_MHz⟩ 다.
서브밴드마다 σ 를 내면 그 다발이 곧 σ(f) 이고, 앵커 문헌의 절차와 같다.

| 대역 B | ΔR = c/2B | Matrice 4E | 여유 | Mini 5 Pro | 여유 |
|---|---|---|---|---|---|
| 400 MHz | 0.375 m | ⚠ 퍼짐 | -0.225 m | ⚠ 퍼짐 | -0.001 m |
| 200 MHz | 0.749 m | 점표적 | +0.150 m | 점표적 | +0.373 m |
| 100 MHz | 1.499 m | 점표적 | +0.900 m | 점표적 | +1.123 m |
| 50 MHz | 2.998 m | 점표적 | +2.399 m | 점표적 | +2.622 m |

출처 ⟨outputs/report06_derived.json : point_target⟩

### §2-4. 자세 통제 — 각도표본이 성기면 로브를 놓친다

방위는 엔코더 턴테이블로 돌리고, 표본 간격은 `λ/4D` 이하로 잡는다.
가장 촘촘한 요구는 1.38° ⟨outputs/report06_derived.json : aspect_finest_deg⟩ (`matrice4e` · `WiFi`)이고, 한 바퀴에 262 ⟨outputs/report06_derived.json : aspect_n_az_max⟩ 표본이다.

⚠ 앵커 문헌은 2° 를 썼고 본인들이 고주파에서 성길 수 있다고 적었다. 같은 선택을 하지 않는다.
로터는 **정지**시키고 블레이드 방위를 기록한다 — 앵커가 회전 성분을 뺐으므로 맞춘다.

| 기체 | 밴드 | Δφ 나이퀴스트 | Δφ 권장 | 한 바퀴 표본수 | 앵커 2° 보다 촘촘 |
|---|---|---|---|---|---|
| matrice4e | LTE | 7.78° | 3.89° | 93 | 아니오 |
| matrice4e | 5G | 4.09° | 2.05° | 176 | 아니오 |
| matrice4e | WiFi | 2.75° | 1.38° | 262 | 예 |
| mini5pro | LTE | 12.39° | 6.20° | 58 | 아니오 |
| mini5pro | 5G | 6.53° | 3.26° | 110 | 아니오 |
| mini5pro | WiFi | 4.38° | 2.19° | 164 | 아니오 |

출처 ⟨outputs/report06_derived.json : aspect⟩

### §2-5. 배경 차감과 지면반사 — 야외는 무향실이 아니다

배경 S_BG 는 **지지대를 세운 채로** 재고 **복소수로** 뺀다.
표적을 경유한 지면반사가 표적 리턴과 같은 거리빈에 들어오면 σ 가 오염된다.
경로차 `2hH/R` 이 서브밴드 거리분해능보다 **커야** 레인지게이팅으로 뗀다.

![ground bounce](outputs/figures/report06_ground_bounce.png)

**그림 4.** 어떤 야외 기하가 지면반사 유령을 표적 거리빈 밖으로 밀어내는가?

27 ⟨outputs/report06_derived.json : ground_bounce_n_geom⟩개 기하 중 200 MHz ⟨outputs/report06_derived.json : ground_bounce_ref_bw_MHz⟩ 서브밴드에서 분리되는 비율은 78% ⟨outputs/report06_derived.json : ground_bounce_sep_frac_200MHz⟩ 다. 안 되면 표적을 더 높이 띄우거나 안테나를 낮춘다.

## §3. 시뮬과 실측 — 무엇이 비교 가능하고 무엇이 아닌가

**구조는 같고 환경은 다르다.** 이 한 줄이 §4 결정표의 마지막 열을 결정한다.

| 같아서 비교 가능한 축 | 달라서 비교 불가능한 축 |
|---|---|
| 바이스태틱 구조 — 기준 1 + 감시 3 | 환경 — 시뮬은 자유공간, 실측은 지면반사·다중경로 |
| 같은 기체 2종(Matrice 4E · Mini 5 Pro) | 클러터 — 시뮬에는 정적 산란체가 없다 |
| 같은 세 파형(LTE · 5G · WiFi) | 동적범위 — 시뮬 ECA 는 무한, 실측은 12-bit |
| 같은 검출기 사슬(ECA → 거리도플러 → CA-CFAR) | 자세 — 시뮬은 각도격자, 비행 중에는 미통제 |
| σ 통계 규약(방위 선형평균) | 절대 탐지거리 — 링크예산의 전제가 다르다 |

실측이 결판낼 수 있는 것은 **σ 자체와 상대 순서**이고, 절대 탐지거리가 아니다.

### §3-1. 앵커가 통제하지 못한 항 — 이 캠페인이 닫으러 가는 목록

| 미통제 항목 | 상태 | 크기 |
|---|---|---|
| polarisation | UNRESOLVED | 미상 |
| statistic convention (Das mu) | RESOLVED_EMPIRICALLY | +0.93 dB |
| size transfer law | UNRESOLVED | +9.50 dB |
| single platform / single lab | UNRESOLVED | 미상 |
| elevation matching | PARTIAL | +0.73 dB |
| near-field vs far-field, environment | OK | +0.00 dB |

출처 ⟨outputs/sigma_anchor.json : uncontrolled⟩

가장 큰 미해소 항(크기전이 법칙)이 실제로 적용된 보정 대부분보다 크다.
그래서 이 캠페인의 첫 표적은 탐지성능이 아니라 **이 원장을 줄이는 것**이다.

| 기체 | 앵커 대비 등급 | 크기비 | L² 보정 | L⁴ 보정 |
|---|---|---|---|---|
| Matrice 4E | scaled | 1.254 ⟨outputs/sigma_anchor.json : drones.matrice4e.comparability.size_ratio⟩ | +1.96 dB ⟨outputs/sigma_anchor.json : drones.matrice4e.comparability.size_corr_L2_db⟩ | +3.93 dB ⟨outputs/sigma_anchor.json : drones.matrice4e.comparability.size_corr_L4_db⟩ |
| Mini 5 Pro | scaled | 0.786 ⟨outputs/sigma_anchor.json : drones.mini5pro.comparability.size_ratio⟩ | -2.09 dB ⟨outputs/sigma_anchor.json : drones.mini5pro.comparability.size_corr_L2_db⟩ | -4.19 dB ⟨outputs/sigma_anchor.json : drones.mini5pro.comparability.size_corr_L4_db⟩ |

등급이 다르면 같은 보정이라도 뜻이 다르다 — 두 기체 다 `scaled` 이지 `direct` 가 아니다.

## §4. ⭐ 결정표 — 어느 측정이 어느 주장을 결판내는가

왼쪽은 시뮬이 지금 하는 주장, 가운데는 그것을 건드리는 측정, 오른쪽은 **판정 임계**다.
임계를 못 맞추면 그 행은 미결로 남는다.

| 02편의 주장 | 결판내는 측정 | 판정 임계 |
|---|---|---|
| 자세에 따른 σ 의 **구조**(로브 위치)는 기하에서 나온다 | 턴테이블 방위컷, 정지 로터 | Δφ ≤ 1.38° ⟨outputs/report06_derived.json : aspect_finest_deg⟩ (§2-4) |
| **절대 레벨은 우리 것이 아니다** — 앵커에서 받는다 | 같은 세션 교정구 + 배경 코히런트 차감 | 교정구 여유 ≥ 3.73 dB ⟨outputs/report06_derived.json : calibration_margin_min_db⟩ (§2-2) |
| **주파수 기울기도 우리 것이 아니다** — PO 에 회절항이 없다 | 세 밴드를 같은 세션에서 측정 | 세션 재현성 < 2.44 dB ⟨outputs/report06_derived.json : slope.gap_db_min⟩ (§4-1) |
| 크기전이 법칙 L² / L⁴ 는 **미해소**다 | 두 기체를 한 캠페인에서 측정 | 차등 4.06 dB ⟨outputs/report06_derived.json : size_law.differential_db⟩ (§4-2) |
| 편파는 **미해소**다 — 커널에 편파 자유도가 없다 | VV / VH / HV / HH 4조합 | 무편파 스칼라 모형의 오차를 처음으로 숫자화 |

| 03~05편의 주장 | 결판내는 측정 | 판정 임계 |
|---|---|---|
| 파형별 점유·대역폭 대가는 σ 와 무관하게 정확하다(03) | X410 이 같은 기하에서 세 파형을 송신 | σ 무관량이라 **반증 대상이 아니다** — 사슬 확인용 |
| 명목 Pfa 를 교정해야 파형 비교가 성립한다(04) | 표적 없는 배경 CPI 를 길게 녹화해 CFAR 임계 넘김을 센다 | 실제 클러터로 교정이 전이되는가 — 통제 Pfa 는 시뮬만 가능 |
| 자유공간에서 어느 조명원이 어느 거리까지 보이나(05) | 야외 고정기하 탐지시험 | **절대 거리는 비교 불가**(§3). 파형 간 순서만 본다 |
| 12-bit ADC 가 직접파 제거의 천장이다(§1) | 직접파를 실제로 받아 ECA 잔차를 잰다 | 여유 19.2 dB ⟨outputs/report06_measurement.json : adc.headroom_db_min⟩ 가 야외에서 얼마나 줄어드는가 |

### §4-1. 기울기 — 세션 재현성이 판정의 문턱이다

![slope](outputs/figures/report06_slope.png)

**그림 5.** 우리 기울기와 앵커 기울기를 가르려면 세션 재현성이 얼마나 좋아야 하는가?

우리 커널은 0.936 ⟨outputs/report06_derived.json : slope.rows[0].ours_db_per_ghz⟩ ~ 1.517 ⟨outputs/report06_derived.json : slope.rows[1].ours_db_per_ghz⟩ dB/GHz 이고 앵커는 0.21 ⟨outputs/report06_derived.json : slope.anchor_db_per_ghz⟩ dB/GHz 다. 대역 3.367 GHz ⟨outputs/report06_derived.json : slope.span_ghz⟩ 를 지나며 두 가설이 2.44 ⟨outputs/report06_derived.json : slope.gap_db_min⟩ ~ 4.40 ⟨outputs/report06_derived.json : slope.rows[1].gap_db⟩ dB 벌어진다.

기울기의 정의는 하나로 고정한다 — 세 밴드 방위평균 μ 를 f[GHz] 에 1차 적합(el=0) ⟨outputs/report06_derived.json : slope.fit_note_short⟩.

### §4-2. 크기법칙 — 두 기체를 함께 사는 이유

![size law](outputs/figures/report06_size_law.png)

**그림 6.** 두 기체를 함께 재면 L² 와 L⁴ 를 가를 수 있는가?

Matrice 4E 는 앵커보다 크고(크기비 1.254 ⟨outputs/report06_derived.json : size_law.by_airframe.matrice4e.size_ratio⟩), Mini 5 Pro 는 작다(0.786 ⟨outputs/report06_derived.json : size_law.by_airframe.mini5pro.size_ratio⟩). 두 법칙의 예측이 **반대 방향**으로 갈리므로 한 대만 재면 레벨 오차와 구별되지 않는다.

차등신호 4.06 dB ⟨outputs/report06_derived.json : size_law.differential_db⟩ 가 두 기체를 함께 사는 이유다.

In [ ]:
# 이 편의 숫자를 직접 열어보기 — 표·그림의 모든 값은 아래 JSON 에서 나온다.
import json
D = json.load(open('outputs/report06_derived.json'))
print(json.dumps(D['_meta']['definitions'], ensure_ascii=False, indent=1))
print('원거리장 채택 :', D['farfield_adopted'])
print('기울기 판별폭 :', [(r['airframe'], round(r['gap_db'], 2))
                          for r in D['slope']['rows']])
print('크기법칙 차등 :', round(D['size_law']['differential_db'], 2), 'dB')

## §5. 이 편의 한계

| 아직 안 되어 있는 것 | 다음 사람이 이어받을 지점 |
|---|---|
| 아무것도 측정하지 않았다 — 이 편은 전부 설계값이다 | 기체 2종 입고 후 §2 의 6조건대로 세션을 돌리고, 산출물을 `outputs/measured_sigma.json` 으로 굳혀 `src/sigma_anchor.py:788` 의 앵커를 교체한다 |
| 장비 선택이 미결이다 — VNA 가 σ(f) 에는 정도이고 X410 은 보유 장비다 | `docs/MEASUREMENT_PLAN.md` §2 의 비교표에서 하나를 고르고, X410 이면 §1-1 교정 절차를 더 엄격히 적용한다 |
| 편파 축이 커널에 없다 — VV 를 재도 붙일 곳이 없다 | `src/materials.py` 의 `gamma_po()` 를 편파 의존 프레넬로 바꿀지 먼저 결정한다 |
| 바이스태틱 측정 설계가 없다 — 이 편의 수치는 전부 모노스태틱 기준이다 | β≤45° 제한 안에서 송수신 분리각별 기하를 §2-1·§2-5 와 같은 방식으로 계산한다 |
| 마이크로도플러 세션이 설계에 없다 | 정지 로터 세션과 **별도**로 회전 세션을 잡는다 — 앵커와의 사과-대-사과를 깨지 않기 위해서다 |
| 지면반사 분리는 평탄지면 2hH/R 근사만 본다 | 실제 부지의 지형·반사체를 넣은 기하로 §2-5 표를 다시 계산한다 |